<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week8/Day2/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Créer un agent avec LangGraph et l'API Gemini

In [1]:
%pip install -qU "langgraph==1.2.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"

In [2]:
import os
from google.colab import userdata

# IMPORTANT: Store your API key in Colab secrets under the name 'GOOGLE_API_KEY'.
# You can obtain a key from https://makersuite.google.com/app/apikey
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [3]:
from google.colab import userdata
userdata.get('secretName')

SecretNotFoundError: Secret secretName does not exist.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class OrderState(TypedDict):
    """State representing the customer's order conversation."""

    # The chat conversation. This preserves the conversation history
    # between nodes. The `add_messages` annotation indicates to LangGraph
    # that state is updated by appending returned messages, not replacing
    # them.
    messages: Annotated[list, add_messages]

    # The customer's in-progress order.
    order: list[str]

    # Flag indicating that the order is placed and completed.
    finished: bool
# The system instruction defines how the chatbot is expected to behave and includes
# rules for when to call different functions, as well as rules for the conversation, such
# as tone and what is permitted for discussion.
BARISTABOT_SYSINT = (
    "system",  # 'system' indicates the message is a system instruction.
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

# This is the message with which the system opens the conversation.
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

# Try using different models. The `pro` models perform the best, especially
# with tool-calling. The `flash` models are super fast, and are a good choice
# if you need to use the higher free-tier quota.
# Check out the features and quota differences here: https://ai.google.dev/pricing
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-latest")

def chatbot(state: OrderState) -> OrderState:
    """The chatbot itself. A simple wrapper around the model's own chat interface."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition.
graph_builder = StateGraph(OrderState)

# Add the chatbot function to the app graph as a node called "chatbot".
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint.
graph_builder.add_edge(START, "chatbot")

chat_graph = graph_builder.compile()

In [ ]:
from pprint import pprint

user_msg = ("user", WELCOME_MSG)
state = chat_graph.invoke({"messages": [user_msg]})

# The state object contains lots of information. Uncomment the pprint lines to see it all.
# pprint(state)

# Note that the final state now has 2 messages. Our HumanMessage, and an additional AIMessage.
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

In [ ]:
user_msg = ("user", "I would like an espresso")

state["messages"].append(user_msg)
state = chat_graph.invoke(state)

# pprint(state)
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

In [ ]:
from langchain_core.messages.ai import AIMessage

def human_node(state: OrderState) -> OrderState:
    """Display the last model message to the user, and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = input("User: ")

    # If it looks like the user is trying to quit, flag the conversation
    # as over.
    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """The chatbot itself. A wrapper around the model's own chat interface."""

    if state["messages"]:
        # If there are messages, continue the conversation with the Gemini model.
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        # If there are no messages, start with the welcome message.
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [new_output]}


# Start building a new graph.
graph_builder = StateGraph(OrderState)

# Add the chatbot and human nodes to the app graph.
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)

# Start with the chatbot again.
graph_builder.add_edge(START, "chatbot")

# The chatbot will always go to the human next.
graph_builder.add_edge("chatbot", "human")

In [ ]:
from typing import Literal

def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


graph_builder.add_conditional_edges("human", maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

# Image(chat_with_human_graph.get_graph().draw_mermaid_png())

In [ ]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    # Note that this is just hard-coded text, but you could connect this to a live stock
    # database, or you could use Gemini's multi-modal capabilities and take live photos of
    # your cafe's chalk menu or the products on the counter and assmble them into an input.

    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate
Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

    "dirty" means add a shot of espresso to a drink that doesn't usually have it, like "Dirty Chai Latte".
    "Regular milk" is the same as 'whole milk'.
    "Sweetened" means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
  """


# Add the new tool to the graph. The `get_menu` tool is embedded in a `ToolNode` which
# handles calling it and passing the response as a message through the graph. Tools are
# also bound to the `llm` object so that the underlying model recognizes them. Since
# you are now using a different `llm` object, you must update the chatbot node to be
# aware of these tools.


from langgraph.prebuilt import ToolNode


# Define the tools and create a "tools" node.
tools = [get_menu]
tool_node = ToolNode(tools)

# Attach the tools to the model so that it knows what it can call.
llm_with_tools = llm.bind_tools(tools)

def maybe_route_to_tools(state: OrderState) -> Literal["tools", "human"]:
    """Route between human or tool nodes, depending if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    # Only route based on the last message.
    msg = msgs[-1]

    # When the chatbot returns tool_calls, route to the "tools" node.
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"

def chatbot_with_tools(state: OrderState) -> OrderState:
    """The chatbot with tools. A simple wrapper around the model's own chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    # Set up some defaults if not already set, then pass through the provided state,
    # overriding only the "messages" field.
    return defaults | state | {"messages": [new_output]}


graph_builder = StateGraph(OrderState)

# Add the nodes, including the new tool_node.
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)

# Chatbot may go to tools, or human.
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human may go back to chatbot, or exit.
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_menu = graph_builder.compile()

# Image(graph_with_menu.get_graph().draw_mermaid_png())

In [ ]:
from collections.abc import Iterable
from random import randint

from langgraph.prebuilt import InjectedState
from langchain_core.messages.tool import ToolMessage

# These functions have no body; LangGraph does not allow @tools to update
# the conversation state, so you will implement a separate node to handle
# state updates. Using @tools is still very convenient for defining the tool
# schema, so empty functions have been defined that will be bound to the LLM
# but their implementation is deferred to the order_node.


@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Returns:
      The updated order in progress.
    """


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""
@tool
def clear_order():
    """Removes all items from the user's order."""


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """

def order_node(state: OrderState) -> OrderState:
    """The ordering node. This is where the order state is manipulated."""
    tool_msg = state["messages"][-1]
    order = state["order"]
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":

            # Each order item is just a string. This is where it assembled as "drink (modifiers, ...)".
            modifiers = tool_call["args"].get("modifiers", [])
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"

            order.append(f'{tool_call["args"]["drink"]} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":

            # We could entrust the LLM to do order confirmation, but it is a good practice to
            # show the user the exact data that comprises their order so that what they confirm
            # precisely matches the order that goes to the kitchen - avoiding hallucination
            # or reality skew.

            # In a real scenario, this is where you would connect your POS screen to show the
            # order to the user.

            print("Your order:")
            if not order:
                print("  (no items)")

            for drink in order:
                print(f"  {drink}")

            response = input("Is this correct? ")

        elif tool_call["name"] == "get_order":

            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":

            order.clear()
            response = None

        elif tool_call["name"] == "place_order":

            order_text = "\n".join(order)
            print("Sending order to kitchen!")
            print(order_text)

            # TODO(you!): Implement cafe.
            order_placed = True
            response = randint(1, 5)  # ETA in minutes

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        # Record the tool results as tool messages.
        outbound_msgs.append(
            ToolMessage(
                content=response,
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}

def maybe_route_to_tools(state: OrderState) -> str:
    """Route between chat and tool nodes if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        # When an order is placed, exit the app. The system instruction indicates
        # that the chatbot should say thanks and goodbye at this point, so we can exit
        # cleanly.
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Route to `tools` node for any automated tool calls first.
        if any(
            tool["name"] in tool_node.tools_by_name.keys() for tool in msg.tool_calls
        ):
            return "tools"
        else:
            return "ordering"

    else:
        return "human"

In [ ]:
# Auto-tools will be invoked automatically by the ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools will be handled by the order node.
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# The LLM needs to know about all of the tools, so specify everything here.
llm_with_tools = llm.bind_tools(auto_tools + order_tools)


graph_builder = StateGraph(OrderState)

# Nodes
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Chatbot -> {ordering, tools, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools (both kinds) always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_order_tools = graph_builder.compile()

# Image(graph_with_order_tools.get_graph().draw_mermaid_png())

In [4]:
display(llm)

NameError: name 'llm' is not defined

# Task
## Task


## Plan
* **Verify LLM Configuration**: Ensure that the `llm` object is correctly initialized and bound with all necessary tools (`auto_tools` and `order_tools`) for the BaristaBot to function properly, referencing cell `0d15e219-5487-4aa7-b50a-f111818ce0e9` and `893c5c16-b51f-4c54-ab18-8f5223e7a00f`.
* **Integrate Order Management Tools**: Confirm that the `order_node` (defined in cell `787f0b54-469b-4351-8597-90924ec14695`) correctly handles the logic for `add_to_order`, `confirm_order`, `get_order`, `clear_order`, and `place_order` tools. Verify that the `maybe_route_to_tools` function properly directs tool calls to either the `tools` node (for `get_menu`) or the `ordering` node (for order management tools).
* **Review Chatbot Logic**: Examine the `chatbot_with_tools` function to ensure it effectively uses the `llm_with_tools` object and processes messages, correctly invoking tools based on the `BARISTABOT_SYSINT` instructions. Pay attention to how it handles `WELCOME_MSG` and subsequent user inputs.
* **Validate Graph Edges and Nodes**: Review the graph structure defined in cell `0d15e219-5487-4aa7-b50a-f111818ce0e9` to confirm that all nodes (`chatbot`, `human`, `tools`, `ordering`) and conditional edges are correctly defined for a seamless conversational flow, including routing to `END` when an order is placed or the user exits.
* **Simulate Conversation Flow**: Outline a series of hypothetical user interactions to test the complete BaristaBot functionality, including asking for the menu, adding items to the order, confirming the order, and placing it. This step will serve as a manual walkthrough to ensure the logic aligns with the intended behavior.
* **Final Task**: Summarize the completed BaristaBot functionality and confirm that all specified sections of the exercise have been addressed without execution.

## Verify LLM Configuration

### Subtask:
Ensure that the `llm` object is correctly initialized and bound with all necessary tools (`auto_tools` and `order_tools`) for the BaristaBot to function properly.


### Verification of LLM Configuration

1.  **LLM Initialization (Cell `893c5c16-b51f-4c54-ab18-8f5223e7a00f`):**
    The `llm` object is correctly initialized using `ChatGoogleGenerativeAI` with the `gemini-2.5-flash-latest` model. This aligns with the requirement for setting up the language model.

2.  **Tool Binding (Cell `0d15e219-5487-4aa7-b50a-f111818ce0e9`):**
    The `llm_with_tools` object is properly created by binding both `auto_tools` (containing `get_menu`) and `order_tools` (containing `add_to_order`, `confirm_order`, `get_order`, `clear_order`, and `place_order`) to the `llm` object. This ensures that the BaristaBot is aware of and can utilize all necessary tools for its operation.

**Conclusion:** The `llm` object is correctly initialized, and all required `auto_tools` and `order_tools` are appropriately bound to `llm_with_tools`. No discrepancies or missing bindings were identified.

## Integrate Order Management Tools

### Subtask:
Confirm that the `order_node` (defined in cell `787f0b54-469b-4351-8597-90924ec14695`) correctly handles the logic for `add_to_order`, `confirm_order`, `get_order`, `clear_order`, and `place_order` tools. Verify that the `maybe_route_to_tools` function properly directs tool calls to either the `tools` node (for `get_menu`) or the `ordering` node (for order management tools).


### Verification of Order Management Tools Integration

1.  **`order_node` Function Analysis (Cell `787f0b54-469b-4351-8597-90924ec14695`):**
    *   **`add_to_order`**: The function correctly extracts the `drink` and `modifiers` from the tool call arguments, formats them into a string (e.g., "Coffee (modifier1, modifier2)"), and appends it to the `state["order"]` list. It then sets the `response` to the joined order items. This is as expected.
    *   **`confirm_order`**: This section simulates a user interaction by printing the current order items and then using `input()` to receive a free-text response from the user. This effectively allows for order confirmation and user feedback, fulfilling its purpose.
    *   **`get_order`**: It retrieves the current `order` list and joins its items with newlines, providing a clear string representation of the order. If the order is empty, it returns "(no order)", which is appropriate.
    *   **`clear_order`**: The `order.clear()` method is used to empty the `order` list, correctly implementing the functionality to remove all items from the customer's order.
    *   **`place_order`**: This part prints the order and sets `order_placed` to `True`. It also generates a random integer for `response`, simulating an Estimated Time of Arrival (ETA). This aligns with the task's requirement to simulate placing an order.
    *   **ToolMessage Creation and State Update**: For each tool call, a `ToolMessage` is correctly created with `content`, `name`, and `tool_call_id`. The `order` list is updated in place, and the `finished` flag is set correctly when `place_order` is called. The `outbound_msgs` are appended to the `messages` in the state.

2.  **`maybe_route_to_tools` Function Analysis (Cell `787f0b54-469b-4351-8597-90924ec14695`):**
    *   **`finished` Flag Handling**: The function correctly checks `state.get("finished", False)`. If `True`, it returns `END`, indicating the conversation concludes after an order is placed, as required.
    *   **Tool Calls Detection**: It correctly identifies if the `msg` (the last message in the state) has `tool_calls` and if `len(msg.tool_calls) > 0`.
    *   **Routing Logic**: When tool calls are detected, it checks if `any` of the tool names in `msg.tool_calls` are present in `tool_node.tools_by_name.keys()`. Given that `tool_node` is initialized with `auto_tools = [get_menu]` in cell `0d15e219-5487-4aa7-b50a-f111818ce0e9`, this condition effectively checks if `get_menu` is being called. If `get_menu` is called, it routes to "tools". Otherwise (if other tool calls exist but are not `get_menu`, meaning they are `order_tools`), it routes to "ordering". This precisely matches the requirement to differentiate between `get_menu` and order management tools.
    *   **Default Routing**: If no tool calls are present in the last message, it correctly routes to "human", allowing the conversation to continue.

**Conclusion:** The `order_node` comprehensively handles all specified order management tools, correctly updating the `OrderState` and generating appropriate `ToolMessage` responses. The `maybe_route_to_tools` function accurately routes tool calls to either the `tools` node (for `get_menu`) or the `ordering` node (for other order management tools), and routes to `END` when an order is finished, ensuring the proper flow of the BaristaBot.

## Review Chatbot Logic

### Subtask:
Examine the `chatbot_with_tools` function to ensure it effectively uses the `llm_with_tools` object and processes messages, correctly invoking tools based on the `BARISTABOT_SYSINT` instructions. Pay attention to how it handles `WELCOME_MSG` and subsequent user inputs.


### Verification of Chatbot Logic (Cell `0d15e219-5487-4aa7-b50a-f111818ce0e9`)

1.  **Usage of `llm_with_tools`:** The `chatbot_with_tools` function correctly uses `llm_with_tools.invoke()`. This ensures that the Language Model is aware of and can call all the bound tools (`auto_tools` and `order_tools`), which is crucial for the BaristaBot's functionality.

2.  **`BARISTABOT_SYSINT` Integration:** The `BARISTABOT_SYSINT` system instruction is consistently passed to the LLM by prepending it to the `state["messages"]` list when `llm_with_tools.invoke()` is called. This ensures that the LLM operates under the defined rules and persona of the BaristaBot throughout the conversation.

3.  **Handling `WELCOME_MSG`:** The function checks if `state["messages"]` is empty. If it is, `new_output` is set to an `AIMessage(content=WELCOME_MSG)`. This ensures that the `WELCOME_MSG` is displayed as the initial greeting when a new conversation starts, as intended.

4.  **Processing Subsequent User Inputs:** The `human_node` (defined in cell `c6225a07-1602-45e0-94d1-0f47e30761e0` and integrated into the graph in cell `0d15e219-5487-4aa7-b50a-f111818ce0e9`) is responsible for taking user input. The `human_node` returns the state with the user's input appended to `messages` as a `("user", user_input)` tuple. The `chatbot_with_tools` function then receives this updated state (which includes the user's input) and passes the complete `state["messages"]` (along with `BARISTABOT_SYSINT`) to the LLM for its next turn. This confirms that subsequent user inputs are correctly processed and incorporated into the conversational context.

**Conclusion:** The `chatbot_with_tools` function is well-structured to effectively utilize the `llm_with_tools` object, enforce `BARISTABOT_SYSINT`, and manage the conversational flow by correctly handling the initial `WELCOME_MSG` and all subsequent user interactions. The logic aligns with the requirements for a functional BaristaBot.

## Validate Graph Edges and Nodes

### Subtask:
Review the graph structure defined in cell `0d15e219-5487-4aa7-b50a-f111818ce0e9` to confirm that all nodes (`chatbot`, `human`, `tools`, `ordering`) and conditional edges are correctly defined for a seamless conversational flow, including routing to `END` when an order is placed or the user exits.


### Verification of Graph Edges and Nodes (Cell `0d15e219-5487-4aa7-b50a-f111818ce0e9`)

1.  **Node Identification:**
    *   `chatbot`: Defined and added as a node, responsible for LLM interaction (`chatbot_with_tools`).
    *   `human`: Defined and added as a node, responsible for user input (`human_node`).
    *   `tools`: Defined and added as a node, handling `auto_tools` (specifically `get_menu`) via `ToolNode`.
    *   `ordering`: Defined and added as a node, handling `order_tools` (e.g., `add_to_order`, `place_order`) via `order_node`.
    All specified nodes are correctly identified and correspond to their intended functionalities.

2.  **Conditional Edges from `chatbot`:**
    *   The `graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)` line correctly routes from the `chatbot` node.
    *   The `maybe_route_to_tools` function (as analyzed in the "Integrate Order Management Tools" subtask) accurately directs to:
        *   `"tools"` if `get_menu` is called.
        *   `"ordering"` if any other order management tool is called.
        *   `"human"` if no tool calls are made.
        *   `END` if `state["finished"]` is `True` (meaning an order has been placed).
    This confirms that the routing logic from the `chatbot` node is correctly implemented for tool calls and conversation flow.

3.  **Conditional Edges from `human`:**
    *   The `graph_builder.add_conditional_edges("human", maybe_exit_human_node)` line correctly routes from the `human` node.
    *   The `maybe_exit_human_node` function (defined in cell `673f8e65-276f-474d-91b7-7e61e69b033e`) accurately directs to:
        *   `END` if the user's input indicates they are quitting (`state.get("finished", False)` is `True`).
        *   `"chatbot"` otherwise.
    This ensures proper handling of user input for continuing or ending the conversation.

4.  **Unconditional Edges:**
    *   `graph_builder.add_edge("tools", "chatbot")`: Correctly ensures that after a `get_menu` tool call, the flow returns to the `chatbot` for further interaction.
    *   `graph_builder.add_edge("ordering", "chatbot")`: Correctly ensures that after an order management action, the flow returns to the `chatbot`.
    *   `graph_builder.add_edge(START, "chatbot")`: Correctly sets the `chatbot` as the starting point of the graph.
    All unconditional edges are correctly defined, ensuring a seamless return to the main conversational logic after tool executions.

**Conclusion:** The graph structure defined in cell `0d15e219-5487-4aa7-b50a-f111818ce0e9` correctly establishes all necessary nodes and conditional/unconditional edges, ensuring a robust and logical conversational flow for the BaristaBot. The routing logic, including the `END` state for order completion or user exit, is accurately implemented.

## Simulate Conversation Flow

### Subtask:
Outline a series of hypothetical user interactions to test the complete BaristaBot functionality, including asking for the menu, adding items to the order, confirming the order, and placing it. This step will serve as a manual walkthrough to ensure the logic aligns with the intended behavior.


### Simulated Conversation Flow

This section outlines a hypothetical conversation with the BaristaBot, demonstrating its functionality from greeting to order placement.

1.  **Start Conversation and Initial Greeting:**
    *   **User Action:** `(Implicit start of conversation)`
    *   **Expected Bot Response:** `WELCOME_MSG` (e.g., "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?")

2.  **Request Menu:**
    *   **User Action:** "Can I see the menu?"
    *   **Expected Bot Response:** The BaristaBot should invoke `get_menu` and display the full menu details, including Coffee Drinks, Coffee Drinks with Milk, Tea Drinks, Tea Drinks with Milk, Other Drinks, and all available Modifiers.

3.  **Add Items to Order:**
    *   **User Action:** "I would like a Latte with oat milk and a double Espresso."
    *   **Expected Bot Response:** The bot should invoke `add_to_order` twice (or once with both items) and confirm the additions, possibly by listing the current order: "Your order so far: Latte (oat milk), Espresso (double)" or similar.
    *   **User Action:** "And a Green Tea."
    *   **Expected Bot Response:** The bot should invoke `add_to_order` and update the order: "Your order so far: Latte (oat milk), Espresso (double), Green Tea."

4.  **Confirm Order:**
    *   **User Action:** "What's my order so far?" or "Confirm my order."
    *   **Expected Bot Response:** The bot should invoke `get_order` and then `confirm_order`. It will display the entire current order and prompt the user for confirmation (e.g., "Your order:
        Latte (oat milk)
        Espresso (double)
        Green Tea
        Is this correct?")
    *   **User Action (Simulation):** `(User types 'Yes, that looks good.')`
    *   **Expected Bot Response:** The bot will acknowledge the confirmation.

5.  **Modify Order (if applicable):**
    *   **User Action (Alternative to step 4 confirmation):** `(User types 'Actually, I want a Triple Espresso instead of a Double, and add vanilla sweetener to the Latte.')`
    *   **Expected Bot Response:** The bot should likely use `clear_order` or modify the existing order items and then `add_to_order` with the corrected/modified items. It should then re-confirm the updated order with the user.

6.  **Place Order:**
    *   **User Action:** "Please place my order."
    *   **Expected Bot Response:** The bot should invoke `place_order`. It will print "Sending order to kitchen!" followed by the order details, and then provide an estimated time: "Your order will be ready in X minutes. Thank you and goodbye!"

7.  **End Conversation:**
    *   **Expected Bot Behavior:** The `place_order` tool sets the `finished` flag to `True`, which will trigger the `maybe_route_to_tools` function to route to `END`, effectively ending the conversation gracefully after the final goodbye message.

## Final Task

### Subtask:
Summarize the completed BaristaBot functionality and confirm that all specified sections of the exercise have been addressed without execution.
